# 畳み込みニューラルネットワークと画像処理

PyTorchによる畳み込みニューラルネットワーク（Convolutional Neural Network：CNN）の記述方法を学習します．

**目標：CNNとMLPの違いを理解**

---

例題では，分類問題（MNISTデータセット）を解きます．  
演習では，分類問題（CIFAR-10データセット）を解きます．


---
## この教材について

「3分で学ぶPyTorch」シリーズの **CNN（畳み込みニューラルネットワーク） 基礎編（第1回）** の演習パートです。

このノートブックは**回答ファイル（ans）**です。全演習の解答が入っています。まずは演習ファイル（task）に挑戦してから参照してください。

この教材と関連記事は note で無料公開しています。
シリーズ一覧: https://note.com/technosend/m/m84d841b6d067

---

## 例題 [MNISTデータセット](https://colab.research.google.com/drive/1N50Ug4U5bMHKSIyK24qWYp68OPFAdGff#scrollTo=MPH-dP3vxbhr&forceEdit=true&sandboxMode=true)

**畳み込み2層・全結合1層のCNNの作成**  
28×28ピクセルのグレースケール画像を入力とし，そのラベル（10出力）を出力する分類問題を解く．

[THE MNIST DATABASE of handwritten digits](http://yann.lecun.com/exdb/mnist/)

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="200" src="https://drive.google.com/thumbnail?id=1egOQqlalPdw6r-93wL5zte3--pVhU_9T&sz=w400">


### 例題1. ライブラリのインポート

深層学習演算ライブラリPyTorchなどのライブラリ，パッケージ，モジュールをインポートする．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=14XdT7XWs6JzTil6JhZZw7aM3VpAxdYH2&sz=w400">



#### 例題1のコード

In [ ]:
# 例題1. ライブラリのインポート

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
# japanize-matplotlib は古いパッケージです。
# 代替: !pip install matplotlib-fontja -q  (2023年以降推奨)
!pip install japanize-matplotlib -q
import japanize_matplotlib

#### 今回使うライブラリ，パッケージ，モジュール一覧


- [torch](https://pytorch.org/docs/stable/torch.html)：多次元テンソルのデータ構造とそのテンソルのための算術演算が組み込まれたパッケージ
- [torch.nn](https://pytorch.org/docs/stable/nn.html)：ニューラルネットワークを定義するためのパッケージ  
nnという略称を与えることが多い
- [torch.nn.functional](https://pytorch.org/docs/stable/nn.functional.html)：様々な関数が含まれるモジュール  
Fという略称を与えることが多い
- [torch.optim](https://pytorch.org/docs/stable/optim.html)：最適化器を宣言するためのパッケージ  
optimという略称を与えることが多い
- [torchvision](https://pytorch.org/vision/stable/index.html)：データセットやニューラルネットワークのモデルが含まれるパッケージ
- [torchvision.transforms](https://pytorch.org/vision/stable/transforms.html)：データセットを整形する関数が含まれるパッケージ  
transformsという略称を与えることが多い
- [matplotlib.pyplot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.html)：グラフや画像の描画を行うモジュール  
pltという略称を与えることが多い
- [numpy](https://numpy.org/doc/stable/user/whatisnumpy.html)：行列演算を行うライブラリ（今回は画像の表示のために使用）  
npという略称を与えることが多い
- [japanize_matplotlib](https://github.com/uehara1414/japanize-matplotlib)：Matplotlibで日本語を表示するためのライブラリ

<font color="blue">【TASK】</font>パッケージをインポートしましょう

### 例題2. ニューラルネットワークの定義

ニューラルネットワーククラスを定義して，そのクラスのインスタンスを宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1jl1W_RW7HrU0ksk1a0XrSq6CyldXF4qZ&sz=w400">

#### 例題2のコード

In [ ]:
# 例題2. ニューラルネットワークの定義

# 1. ニューラルネットワーククラスの定義
class GrayImageClassifier(nn.Module):
    def __init__(self):
        super(GrayImageClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 20, 5)
        self.conv2 = nn.Conv2d(20, 20, 5)
        self.fc1 = nn.Linear(320, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.max_pool2d(x, 2) 
        x = F.relu(x)

        x = self.conv2(x)
        x = F.avg_pool2d(x, 2) 
        x = F.relu(x)

        x = x.view(-1, 320)

        x = self.fc1(x)
        return x

# 2. インスタンスの宣言
gray_image_classifier = GrayImageClassifier()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gray_image_classifier.to(device)

#### 1. ニューラルネットワーククラスの定義  
  - nn.Moduleを継承したクラス「GrayImageClassifier」を定義
  - [\_\_init\_\_()とforward()を定義](https://drive.google.com/file/d/115-MoDnZXSpyYa0qekNzl8m2UhHz8yu4/view?usp=sharing)
    ```python
    class GrayImageClassifier(nn.Module):
        def __init__(self):
            super(GrayImageClassifier, self).__init__()
            # 【TASK】畳み込み層と全結合層を宣言
        def forward(self, x):
            # 【TASK】順伝播のパスを定義
            return x
    ```


  - \_\_init\_\_()では，畳み込み層と全結合層を宣言
    - [super()](https://docs.python.org/ja/3/library/functions.html#super)を呼び出す
    - 一つの層につき一つ，nnパッケージ内のクラスのインスタンスを宣言するので，今回は三つ宣言
    - <font color="red">【NEW!】</font>畳み込み層は[nn.Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)クラスを用いて宣言
    ```python
    self.conv1 = nn.Conv2d(1, 20, 5, stride=1, padding=0)
    # 第１引数：入力チャンネル（int）
    # 第２引数：出力チャンネル（int）
    # 第３引数：カーネル幅（int）
    # 第４引数（オプション）：ストライド幅（int），デフォルト1
    # 第５引数（オプション）：パディング幅（int），デフォルト0
    ```
    - 全結合層は[nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear)クラスを用いて宣言  
    ```python
    self.fc1 = nn.Linear(320, 10)
    ```
    <font color="blue">【TASK】</font>畳み込み層と全結合層をnn.Conv2dクラスとnn.Linearクラスを用いて宣言しましょう  
    ニューラルネットワークの構成は下記の通りです  
        - 畳み込み層１：入力1 [ch]，出力20 [ch]，カーネル幅5 [pixel]
        - 畳み込み層２：入力20 [ch]，出力20 [ch]，カーネル幅5 [pixel]
        - 全結合層１：入力320，出力10　※入力が320になる理由は後で説明

  - forward()では，順伝播のパスを定義  
    - forward()は外部からの入力データxを受け取り，順伝播を行う  
    - nn.Linearクラスのインスタンスに()をつけて引数を与えて呼び出すと，クラスメソッドである\_\_call_\_()が呼び出され，引数に対する全結合層の計算結果が返される
    - <font color="red">【NEW!】</font>nn.Conv2dクラスのインスタンスに()をつけて引数を与えて呼び出すと，クラスメソッドである\_\_call_\_()が呼び出され，引数に対する畳み込み層の計算結果が返される  
        ```python
        x = self.conv1(x) 
        x = self.fc1(x)
        ```

    - <font color="red">【NEW!】</font>畳み込み層と全結合層が接続している場合，データの形状を変更する必要がある
        - [torch.Tensor](https://pytorch.org/docs/stable/tensors.html#torch.Tensor)型（以降，Tensor型）のデータは[torch.view()](https://pytorch.org/docs/stable/generated/torch.Tensor.view.html)を使ってデータの形状を変更できる
    ```python
    # 畳み込み層の出力は(バッチ数, チャンネル数, 縦幅, 横幅)
    x = ...
    # 全結合層の入力に合わせて(バッチ数, チャンネル数×縦幅×横幅)の形状に変更
    x = x.view(-1, 320)
    x = self.fc1(x)
    ```
    - <font color="red">【NEW!】</font>最大プーリングは[nn.functional.max_pool2d](https://pytorch.org/docs/stable/generated/torch.nn.functional.max_pool2d.html)クラス，平均プーリングは[nn.functional.avg_pool2d](https://pytorch.org/docs/stable/generated/torch.nn.functional.avg_pool2d.html)クラスをそれぞれ用いて呼び出す
    ```python
    x = F.max_pool2d(x, 2, stride=None, padding=0)
    x = F.avg_pool2d(x, 2, stride=None, padding=0)
    # 第１引数：入力（torch.Tensor）
    # 第２引数：カーネル幅（int）
    # 第３引数（オプション）：ストライド幅（int），デフォルトNone
    # ※Noneにするとカーネル幅になる
    # ※strideのデフォルト値が畳み込み，Conv系と違うので注意しましょう
    # 第４引数（オプション）：パディング幅（int），デフォルト0
    ```
    - ReLU関数は[nn.functional.relu](https://pytorch.org/docs/stable/generated/torch.nn.functional.relu.html#torch.nn.functional.relu)クラスを用いて呼び出す
    - 中間層にはReLU関数が必要
    ```python
    x = F.relu(x)
    ```
    <font color="blue">【TASK】</font>順伝播のパスを定義しましょう  
    28×28ピクセルのグーレースケール画像を入力すると想定したとき順伝播のパスの構成は下記の通りです  
        1. 畳み込み層１  
        2. 最大プーリング１：カーネル幅2
        3. ReLU関数
        4. 畳み込み層２
        5. 平均プーリング２：カーネル幅2
        6. ReLU関数
        7. テンソル形状変更：入力：20×4×4，出力：320
        8. 全結合層１



#### 2. インスタンスの宣言

- gray_image_classifierという名前でインスタンスを宣言
    ```python
    gray_image_classifier = # 【TASK】ニューラルネットワーククラスのインスタンスの宣言
    ```

- GPUにセットアップ

    ```python
    device = # 【TASK】GPUの指定
    # 【TASK】GPUにセットアップ
    ```

- ニューラルネットワーククラスのインスタンスの宣言

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスを宣言しましょう 

- GPUにセットアップ
  - Moduleクラスにもto()がある
  - Tensor型変数同様にto()を使ってGPUにデータを渡すことができる
  - 学習時にGPUを使う場合は，ニューラルネットワーククラスのインスタンスをGPUに渡す

  <font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスをGPUにセットアップしましょう  
  - `torch.cuda.is_available()` でGPU有無を判定し，利用可能な場合は"cuda:0"，利用不可の場合は"cpu"を指定しましょう
  - [to()](https://pytorch.org/docs/1.9.1/generated/torch.Tensor.to.html)を使ってGPUにセットアップしましょう


### 例題3. 誤差関数・最適化器の設定

誤差関数と最適化器を宣言する．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1oS8_oitSvcQ9f5_uYMrDXyC3P_vrM7jp&sz=w400">



#### 例題3のコード

In [ ]:
# 例題3. 誤差関数・最適化器の設定

# 1. 誤差関数の宣言
criterion = nn.CrossEntropyLoss()

# 2. 最適化器の宣言
optimizer_gray_image_classifier = optim.SGD(gray_image_classifier.parameters(), lr=0.01)

#### 1. 誤差関数の宣言

- クロスエントロピー誤差を計算するcriterionを宣言

    ```python
    criterion = # 【TASK】誤差関数の宣言
    ```


- クロスエントロピー誤差関数は[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)クラスを用いて宣言
- nn.CrossEntropyLoss()で宣言したインスタンスは，二つの引数のクロスエントロピー誤差を返す  
  ```python
  criterion = nn.CrossEntropyLoss()
  ```

<font color="blue">【TASK】</font>誤差関数を宣言しましょう
- クロスエントロピー誤差関数を使いましょう
- nn.CrossEntropyLossクラスを用いて宣言しましょう

#### 2. 最適化器の宣言  

- 確率的勾配降下法を計算するoptimizer_gray_image_classifierを宣言

    ```python
    optimizer_gray_image_classifier = # 【TASK】最適化器の宣言
    ```

- 確率的勾配降下法は[optim.SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html#torch.optim.SGD)クラスを用いて宣言  
    ```python
    optimizer_gray_image_classifier = optim.SGD(gray_image_classifier.parameters(), lr=0.01)
    # 第1引数：ニューラルネットワークのパラメータ(nn.Modules.parameters())
    # 第2引数：学習率(float)
    ```

<font color="blue">【TASK】</font>最適化器を宣言しましょう  
- 確率的勾配降下法を使いましょう
- optim.SGDクラスを用いて宣言しましょう
- 引数の構成は下記の通りです  
  - ニューラルネットワークのパラメータgray_image_classifierのパラメータ
  - lr：学習率0.01

### 例題4. データセットの準備

MNISTデータセットを読み込み，データの整形やミニバッチの設定，データローダーの作成を行う．

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=17fS-oMI83rSL7SxN_GyKHjxP-FO-R3aQ&sz=w400">

#### 例題4のコード

In [ ]:
# 例題4. データセットの準備  

# 1. データ整形の設定  
transform = transforms.Compose([
                                transforms.ToTensor()  # torchvision 0.18以降非推奨,
                                transforms.Normalize((0.5,), (0.5,))
                                ])

# 2. データセットの読み込み
train_set = torchvision.datasets.MNIST(root='./', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./', train=False, download=True, transform=transform)

# 3. ミニバッチの設定とデータローダーの作成
batch_size = 4
train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

#### 1. データ整形の設定

- データ整形の方法を指定するtransformを宣言
```python
transform = # 【TASK】データ整形の方法
```

  - transformという変数名で，[transforms.Compose](https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.Compose)クラスのインスタンスを宣言
  - transforms.Composeクラスのコンストラクタの引数には整形の指示をlist形式で与える
  - データをPyTorchで扱えるようにする[transforms.ToTensor](https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.ToTensor)クラスと平均値と標準偏差値を指示できる[transforms.Normalize](https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.Normalize)クラスを与える
  - transforms.Normalizeクラスには引数として平均値，標準偏差値のタプルを与える  

    ```python
    transform = transforms.Compose([
                                    transforms.ToTensor(),
                                    transforms.Normalize((0.5,), (0.5,))
                                    ])
    ```

<font color="blue">【TASK】</font>データ整形の方法を指定しましょう  
引数の構成は下記の通りです  
- ComposeクラスにはToTensorクラスとNormalizeクラスを与える
- 平均値と標準偏差値はどちらも0.5


#### 2. データセットの読み込み  
- train_setという名前で学習データ用の変数を宣言
- test_setという名前でテストデータ用の変数を宣言
```python
train_set = # 【TASK】学習データの読み込み
test_set = # 【TASK】テストデータの読み込み
```

  - MNISTデータセットの読み込みは[torchvision.datasets.MNIST](https://pytorch.org/vision/stable/datasets.html#mnist)クラスを用いて行う  
  ```python
train_set = torchvision.datasets.MNIST(root="./", train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root="./", train=False, download=True, transform=transform)
  # 第1引数：読み込むデータのディレクトリの指定(str)
  # 第2引数：学習用かテスト用か(bool)
  # 第3引数：ダウンロードするかどうか(bool)
  # 第4引数：データ整形の設定transform(torchvision.transforms.Compose())
  ```

<font color="blue">【TASK】</font>データセットの読み込みを行いましょう  
読み込みの設定は下記の通りです  
- 学習データ
    - ディレクトリ："./"
    - 学習用のデータを選択
    - ダウンロードを行う
    - データ整形は上で宣言したtransformを使う
- テストデータ
    - ディレクトリ："./"
    - テスト用のデータを選択
    - ダウンロードを行う
    - データ整形は上で宣言したtransformを使う

#### 3. ミニバッチの設定とデータローダーの作成  

- 任意のミニバッチの設定でデータを読み込むデーターローダーを作成
- batch_sizeという名前でバッチサイズ用の変数を宣言  
  ※バッチサイズはミニバッチ1つに含まれるデータの数
- train_loaderという名前で学習データのデータローダー用の変数を宣言
- test_loaderという名前でテストデータのデータローダー用の変数を宣言  
```python
batch_size = # 【TASK】バッチサイズ
train_loader = # 【TASK】学習データのデータローダー
test_loader = # 【TASK】テストデータのデータローダー
```

- データローダーの作成は[torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)クラスを用いて行う

    ```python
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)
    # 第1引数：読み込むデータ(Dataset)
    # 第2引数：バッチサイズ(int)
    # 第3引数：シャッフルするかどうか(bool)
    ```

<font color="blue">【TASK】</font>ミニバッチを設定してデータローダーを作成しましょう
- バッチサイズ4
- 学習データのデータローダー
    - 読み込むデータtrain_set
    - シャッフルする
- テストデータのデータローダー
    - 読み込むデータtest_set
    - シャッフルしない



#### おまけ：データの可視化


In [ ]:
# おまけ：データの可視化

# batch_size分の画像（images）と正解ラベル（labels）を取得
images, labels = next(iter(train_loader))

# データをグリッド状に並べ替え
img = torchvision.utils.make_grid(images, nrow=batch_size)

# 値域を修正
img = img / 2 + 0.5

# numpy.ndarray型に変換して表示
npimg = img.numpy()
npimg = np.transpose(npimg, (1, 2, 0))

# 画像を表示
plt.imshow(npimg)
plt.show()

# ラベルを出力
print(labels)

- batch_size分の画像と正解ラベルを取得
    - DataloaderオブジェクトにPythonの組み込み関数[iter()](https://docs.python.org/ja/3/library/functions.html?highlight=iter#iter)を適用するといくつかにまとまった状態で（＝ミニバッチごとに）データを引き出せる
    - 同じくPythonの組み込み関数[next()](https://docs.python.org/ja/3/library/functions.html?highlight=iter#next)を使うことで，ミニバッチごとにデータを取り出せる
    ```python
    images, labels = next(iter(train_loader))
    ```
- データをグリッド状に並べ替え
    - ミニバッチごとに取り出したデータは当然画像ごとに分かれている
    - [torchvision.utils.make_grid()](https://pytorch.org/vision/stable/utils.html#torchvision.utils.make_grid)を用いて，指定した並べ方で一枚の画像にすることができる
    ```python
    img = torchvision.utils.make_grid(images, nrow=batch_size)
    # 第1引数：データ(Tensor)
    # 第2引数：一行あたりの枚数(int)
    ```

- 値域を修正
    - 平均0.5，標準偏差0.5で正規化しているため値域は-1～1
    - Matplotlibでは0～1の値を使うので，嵩上げで値域をずらす
    ```python
    img = img / 2 + 0.5
    ```
- [numpy.ndarray](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html)型に変換 
    - Tensor型はMatplotlibでは読み込めないのでnumpy.ndarray型に変換
    - 縦，横，チャンネルの3軸の並びをMatplotlib形式に揃える
    ```python
    npimg = img.numpy()
    npimg = np.transpose(npimg, (1, 2, 0))
    ```
- 画像の表示
    - 画像の描画には[plt.imshow()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)を使う
    - 描画した画像を表示するには[plt.show()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.show.html)を使う
    ```python
    plt.imshow(npimg)
    plt.show()
    ```

- ラベルを表示
    ```python
    print(labels)
    ```
<font color="blue">【TASK】</font>実行してデータを可視化してみましょう

### 例題5. 学習

教師データとの誤差を計算し，パラメータを更新する．  
学習時の誤差とテスト時の誤差を表示し，テスト時の予測精度を表示する．  

---

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1A6TjumoBejWpKGvD6TDQeivN1tpeEBp4&sz=w400">


 #### 例題5のコード

In [ ]:
# 例題5. 学習

# 1. 学習ループの作成
epochs = 4

# 誤差をグラフ描画するための変数
train_y_axis = []
test_y_axis = []

# エポックのループ
for epoch in range(epochs):
    print("epoch：{}".format(epoch + 1))
    # 学習誤差を確認するための変数の初期化
    train_minibatch_loss = 0.0
    train_epoch_loss = 0.0

    # 学習データのデータローダーのループ
    for data in train_loader:
        # 2. ニューラルネットワークへのデータ入力
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = gray_image_classifier(inputs)

        # 3. 誤差逆伝播とパラメータの更新
        optimizer_gray_image_classifier.zero_grad()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_gray_image_classifier.step()

        # 4. エポックごとの誤差の計算
        train_minibatch_loss += loss.item()
    train_epoch_loss = train_minibatch_loss / len(train_loader)
    print("学習誤差：{}".format(train_epoch_loss))


    # テスト誤差を確認するための変数の初期化
    test_minibatch_loss = 0.0
    test_epoch_loss = 0.0
    # 予測精度を計算するための変数の初期化
    correct = 0.0

    # テストデータのデータローダーのループ
    for data in test_loader:
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = gray_image_classifier(inputs)
        # 4. エポックごとの誤差の計算
        test_minibatch_loss += criterion(outputs, labels).item()

        # 5. 予測精度の計算と表示
        pred = outputs.argmax(dim=1)
        correct += pred.eq(labels).sum().item()

    # 4. エポックごとの誤差の計算
    test_epoch_loss = test_minibatch_loss / len(test_loader)
    print("テスト誤差：{}".format(test_epoch_loss))
    # 5. 予測精度の計算と表示
    print("予測精度：{}%\n".format(correct / len(test_set) * 100))

    # 誤差をグラフ描画するための変数に格納
    train_y_axis.append(train_epoch_loss)
    test_y_axis.append(test_epoch_loss)

# 6. 誤差のグラフ描画
plt.xticks(range(1, len(train_y_axis)+1))
plt.ylim(0, 1)
plt.plot(range(1, len(train_y_axis)+1), train_y_axis, label="学習誤差")
plt.plot(range(1, len(test_y_axis)+1), test_y_axis, label="テスト誤差")
plt.title("学習誤差とテスト誤差")
plt.legend()
plt.show()

#### 1. 学習ループの作成  
- ミニバッチ学習を行う学習ループの作成
- epochsという名前でエポック用の変数を宣言
- 外側にエポック，内側に学習データ・テストデータのデータローダーのループをそれぞれ作成
```python
epochs = # 【TASK】エポック数
for epoch in # 【TASK】エポック
      for data in # 【TASK】学習データのデータローダー
      for data in # 【TASK】テストデータのデータローダー
```

<font color="blue">【TASK】</font>学習ループを作成しましょう  
ループの設定は下記の通りです  
  - エポック
    - エポック数2
    - [range](https://docs.python.org/ja/3/library/stdtypes.html#range)クラスを使ってループさせましょう
  - 学習データのデータローダー
    - ループ対象：学習データのデータローダーtrain_loader
    - [for](https://docs.python.org/ja/3/reference/compound_stmts.html#for)を使ってtrain_loaderをループさせましょう
  - テストデータのデータローダー
    - ループ対象：テストデータのデータローダーtest_loader
    - forを使ってtest_loaderをループさせましょう
    

#### 2. ニューラルネットワークへのデータの入力  
- GPUにセットアップ
- outputsという名前で出力用の変数を宣言
```python
# 画像と正解ラベルに分割
inputs, labels = data
inputs = # 【TASK】GPUにセットアップ
labels = # 【TASK】GPUにセットアップ
outputs = # 【TASK】ニューラルネットワークからの出力
```

  - 学習用ミニバッチのループで取得したデータを画像（inputs）と正解ラベル/教師データ（labels）に分割  
    
<font color="blue">【TASK】</font>ニューラルネットワークへデータを入力しましょう  
  - 入力inputsをGPUにセットアップしましょう
  - 教師データlabelsをGPUにセットアップしましょう
  - inputsをニューラルネットワークに与えて出力outputsを取得しましょう

#### 3. 誤差逆伝播とパラメータの更新  
- パラメータの微分値を初期化
- loss という名前で誤差計算の結果用の変数を宣言
- 誤差逆伝播
- パラメータの更新
```python
# 【TASK】パラメータの微分値を初期化
loss = # 【TASK】誤差の計算
# 【TASK】誤差逆伝播
# 【TASK】パラメータの更新
```



- パラメータの微分値の初期化は最適化器が持つ
  [zero_grad()](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html#torch.optim.Optimizer.zero_grad)を呼び出して行う
- 誤差計算はnn.MSELossクラスやnn.CrossEntropyLossクラスなどの誤差関数クラスのインスタンスに引数を2つ与えて行う
```python
loss = criterion(outputs, labels)
# 第1引数：ニューラルネットワークの出力
# 第2引数：教師データ
```

- 誤差逆伝播は計算結果を持った変数から[backward()](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html)を呼び出して行う
- パラメータの更新は最適化器の持つ[step()](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html#torch.optim.Optimizer.step)を呼び出して行う  

<font color="blue">　【TASK】</font>誤差逆伝播とパラメータの更新を行いましょう
- zero_grad()を使ってパラメータの微分値の初期化をしましょう
- 誤差の計算結果をlossに与えましょう
- backward()を使って誤差逆伝播させましょう
- step()を使ってパラメータの更新を行いましょう






#### 4. <font color="red">【NEW】</font>エポックごとの誤差の計算

- ミニバッチごとの誤差の平均値を学習，テストそれぞれで計算
- エポックごとの誤差の平均値を学習，テストそれぞれで計算
```python
# 【TASK】学習時のミニバッチごとの平均値を全て足す
train_epoch_loss = # 【TASK】エポックごとの平均値を計算
# 【TASK】テスト時のミニバッチごとの平均値を全て足す
test_epoch_loss = # 【TASK】エポックごとの平均値を計算
```
※テスト側の記述は飛び石になっているので注意しましょう

- train_epoch_loss，test_epoch_lossという名前で学習時，テスト時のエポックごとの誤差計算の結果用の変数をそれぞれ宣言
- train_minibatch_loss，test_minibatch_lossという名前で学習時，テスト時のミニバッチごとの誤差計算の結果用の変数をそれぞれ宣言
- Tensor型データは，データが単体の場合は[item()](https://pytorch.org/docs/stable/generated/torch.Tensor.item.html)，複数の場合は[tolist()](https://pytorch.org/docs/stable/generated/torch.Tensor.tolist.html)をそれぞれ使って，Tensorクラスのアトリビュートやクラスメソッドなどを除いた生のデータを取得
- lossには誤差計算の結果をミニバッチごとに平均した値が一つ格納
- エポックごとの平均値は，そのエポック内のミニバッチごとの平均値を全て足して，イテレーション数で割ると得られる
- イテレーション数はそれぞれtrain_loaderやtest_loaderの長さから取得
```python
# 学習用ミニバッチループの中の処理
train_minibatch_loss += loss.item()
# エポックループの中の処理
train_epoch_loss = train_minibatch_loss / len(train_loader)
```

<font color="blue">【TASK】</font>学習，テストそれぞれのエポックごとの誤差計算の結果を求めましょう



#### 5. <font color="red">【NEW】</font>予測精度の計算と表示
- テストデータを入力したときの出力から予測結果を取得
- 一致している数を集計
- 予測精度を計算
```python
pred = # 【TASK】予測結果の取得
correct += # 【TASK】一致数の集計
```


- outputsの最大値のインデックス（＝予測結果）を取得
    - predという名前で予測結果用の変数を宣言
    - Tensor型データの最大値のインデックスは[torch.argmax()](https://pytorch.org/docs/stable/generated/torch.argmax.html)を使って取得
    ```python
    pred = outputs.argmax(dim=1)
    # 第1引数：最大値を計測する次元の方向(int)
    ```
    <font color="blue">【TASK】</font>CNNの出力から予測結果を取得しましょう

- 教師データと予測ラベルが一致している数を正解数として取得
    - correctという名前で宣言済みの変数に正解数を加算する
    - Tensor型データが一致しているかどうかは[torch.eq()](https://pytorch.org/docs/stable/generated/torch.eq.html)を使って取得
    - Tensor型データの集計は[torch.sum()](https://pytorch.org/docs/stable/generated/torch.sum.html)を使って取得
    - torch.item()を使って生のデータを取得
    - ミニバッチごとの一致数を集計
    ```python
    correct += pred.eq(labels).sum().item()
    # eqの第1引数：比較対象(Tensor)
    ``` 
    <font color="blue">【TASK】</font>予測結果と教師データから正解数を取得しましょう

- 予測精度を計算
    - 一致数をテストデータ全体で割って，精度を計算
    ```python
    correct / len(test_set) * 100
    ```

#### 6. <font color="red">【NEW!】</font>誤差のグラフ描画
- 学習誤差とテスト誤差のグラフを描画
```python
plt.xticks(range(1, len(train_y_axis)+1))
plt.ylim(0, 1)
# 【TASK】学習誤差を描画
# 【TASK】テスト誤差を描画
# 【TASK】タイトルを設定
plt.legend()
plt.show()
```

- x軸の目盛幅の設定には[plt.xticks()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xticks.html)を使う（y軸は[yticks()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.yticks.html)）
```python
plt.xticks(range(1, len(train_y_axis)+1))
# 第1引数：軸の値(list)
```
- y軸の目盛の最大値と最小値の設定には[plt.ylim()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.ylim.html)を使う（x軸は[xlim()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.xlim.html)）
```python
plt.ylim(0, 1)
# 第1引数：最小値(int)
# 第2引数：最大値(int)
```
- 散布図の描画には[plt.plot()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)を使う
```python
plt.plot(range(1, len(train_y_axis)+1), train_y_axis, label="学習誤差")
# 第1引数：x軸の値(list)
# 第2引数：y軸の値(list)
# 第3引数：凡例(str)
```
- タイトルの設定には[plt.title()](https://matplotlib.org/3.1.1/api/_as_gen/matplotlib.pyplot.title.html)を使う
```python
plt.title("学習誤差とテスト誤差")
# 第1引数：タイトル(str)
```
- タイトルと凡例の表示には[plt.legend()](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.legend.html)を使う
- グラフの表示にはplt.show()を使う

<font color="blue">【TASK】</font>誤差をグラフ描画してみましょう  
グラフの設定は下記の通りです
- 学習誤差とテスト誤差のために2系列描画します
- 学習誤差はtrain_y_axisに，テスト誤差はtest_y_axisにそれぞれ格納されています
- x軸のデータはエポック数：1〜4
- y軸のデータは学習誤差とテスト誤差
- 学習誤差の凡例は"学習誤差"，テスト誤差の凡例は"テスト誤差"
- グラフのタイトルは"学習誤差とテスト誤差"
